# Specificity evidence and backends

Small panels typically need an API request per candidate; genome-scale
designs need bulk local execution. Both implement the same protocol and are
cached by sequence plus database and method versions.

In [ ]:
#| default_exp specificity

In [ ]:
#| export
from __future__ import annotations

from dataclasses import asdict, dataclass
from typing import Protocol, Sequence

In [ ]:
#| export
@dataclass(frozen=True)
class SpecificityRequest:
    sequence: str
    target_id: str | None
    database_id: str
    database_version: str
    method_id: str

    @property
    def cache_key(self) -> str:
        return "|".join((self.sequence.upper(), self.target_id or "", self.database_id, self.database_version, self.method_id))

In [ ]:
#| export
@dataclass(frozen=True)
class SpecificityEvidence:
    request: SpecificityRequest
    target_raw_score: float
    best_offtarget_raw_score: float
    off_target_count: int
    best_offtarget_id: str | None = None

    def __post_init__(self) -> None:
        if self.target_raw_score <= 0:
            raise ValueError("target_raw_score must be positive")
        if self.best_offtarget_raw_score < 0 or self.off_target_count < 0:
            raise ValueError("off-target evidence cannot be negative")

    @property
    def specificity_percent(self) -> float:
        """100 means no competing hit; 0 means a competing hit equals target."""
        fraction = min(1.0, self.best_offtarget_raw_score / self.target_raw_score)
        return round(100 * (1 - fraction), 3)

    def as_dict(self) -> dict:
        result = asdict(self)
        result["specificity_percent"] = self.specificity_percent
        return result

In [ ]:
#| export
class SpecificityBackend(Protocol):
    """A remote API or local batch tool that evaluates a request."""
    def evaluate(self, request: SpecificityRequest) -> SpecificityEvidence: ...

In [ ]:
#| export
class SpecificityCache:
    """Minimal in-memory cache; a SQLite/Redis adapter can implement the same API."""
    def __init__(self) -> None:
        self._items: dict[str, SpecificityEvidence] = {}

    def get(self, request: SpecificityRequest) -> SpecificityEvidence | None:
        return self._items.get(request.cache_key)

    def put(self, evidence: SpecificityEvidence) -> SpecificityEvidence:
        self._items[evidence.request.cache_key] = evidence
        return evidence

In [ ]:
#| export
class SpecificityService:
    """Cache-first service suitable for a panel API or a bulk local backend."""
    def __init__(self, backend: SpecificityBackend, cache: SpecificityCache | None = None):
        self.backend, self.cache = backend, cache or SpecificityCache()

    def evaluate(self, request: SpecificityRequest) -> tuple[SpecificityEvidence, bool]:
        cached = self.cache.get(request)
        if cached is not None:
            return cached, True
        return self.cache.put(self.backend.evaluate(request)), False

    def evaluate_many(self, requests: Sequence[SpecificityRequest]) -> list[tuple[SpecificityEvidence, bool]]:
        return [self.evaluate(request) for request in requests]